<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session13_transformer_from_scratch_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۳: Transformer از صفر

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session13_transformer_from_scratch.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import math
import time
import urllib.request
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
BLOCK_SIZE, BATCH_SIZE = 128, 64
MAX_STEPS = 3000 if device.type == 'cuda' else 300

URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
shakespeare = urllib.request.urlopen(URL).read().decode('utf-8')


class CharCorpus:
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}
        data = torch.tensor([self.stoi[c] for c in text], dtype=torch.long)
        n = int(0.9 * len(data))
        self.train, self.val = data[:n], data[n:]

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return ''.join(self.chars[i] for i in ids)

    def get_batch(self, split):
        src = self.train if split == 'train' else self.val
        ix = torch.randint(len(src) - BLOCK_SIZE - 1, (BATCH_SIZE,))
        x = torch.stack([src[i:i + BLOCK_SIZE] for i in ix])
        y = torch.stack([src[i + 1:i + BLOCK_SIZE + 1] for i in ix])
        return x.to(device), y.to(device)


corpus = CharCorpus(shakespeare)


def scaled_dot_product_attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    return torch.softmax(scores, dim=-1) @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, fused=False):
        super().__init__()
        self.n_heads, self.d_head, self.fused, self.p = n_heads, d_model // n_heads, fused, dropout
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = [t.view(B, T, self.n_heads, self.d_head).transpose(1, 2) for t in self.qkv(x).split(C, dim=2)]
        if self.fused:          # PyTorch kernel (Flash / memory-efficient attention on GPU)
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.p if self.training else 0.0)
        else:
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
            out = scaled_dot_product_attention(q, k, v, mask)
        return self.dropout(self.proj(out.transpose(1, 2).contiguous().view(B, T, C)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, fused=False):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout, fused)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


def sinusoidal_encoding(max_len, d):
    pos = torch.arange(max_len).unsqueeze(1)
    div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
    pe = torch.zeros(max_len, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe


class MiniGPT(nn.Module):
    def __init__(self, vocab, d_model=192, n_heads=6, n_layers=6, block_size=BLOCK_SIZE, dropout=0.1,
                 positional='learned', fused=False):
        super().__init__()
        self.block_size, self.positional = block_size, positional
        self.tok_emb = nn.Embedding(vocab, d_model)
        if positional == 'learned':
            self.pos_emb = nn.Embedding(block_size, d_model)
        else:                                             # fixed: a buffer moves with .to(device) but is not trained
            self.register_buffer('pos_enc', sinusoidal_encoding(block_size, d_model))
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(d_model, n_heads, dropout, fused) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        pos = self.pos_emb(torch.arange(T, device=idx.device)) if self.positional == 'learned' else self.pos_enc[:T]
        logits = self.head(self.ln_f(self.blocks(self.drop(self.tok_emb(idx) + pos))))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, top_p=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                kth = torch.topk(logits, top_k).values[:, [-1]]
                logits[logits < kth] = float('-inf')
            if top_p is not None:
                logits = top_p_filter(logits, top_p)
            idx = torch.cat([idx, torch.multinomial(torch.softmax(logits, dim=-1), 1)], dim=1)
        return idx


def top_p_filter(logits, p):
    """Keep the smallest set of tokens whose cumulative probability reaches p (nucleus sampling)."""
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    cum = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
    remove = cum - torch.softmax(sorted_logits, dim=-1) >= p      # tokens *after* the nucleus is complete
    sorted_logits[remove] = float('-inf')
    return torch.full_like(logits, float('-inf')).scatter(-1, sorted_idx, sorted_logits)


def train_gpt(model, corpus, steps=MAX_STEPS, lr=1e-3, warmup=100, log=True):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, steps - warmup))))
    start = time.time()
    for step in range(steps + 1):
        if log and step % (steps // 5) == 0:
            print(f'step {step:5d} | val loss {val_loss(model, corpus):.3f} | {time.time() - start:.0f}s')
        x, y = corpus.get_batch('train')
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return time.time() - start


@torch.no_grad()
def val_loss(model, corpus, n_batches=20):
    model.eval()
    torch.manual_seed(123)                                   # the same validation batches for every model
    loss = torch.stack([model(*corpus.get_batch('val'))[1] for _ in range(n_batches)]).mean().item()
    model.train()
    return loss


def n_params(model):
    return sum(p.numel() for p in model.parameters())

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

به‌جای `scaled_dot_product_attention` خودمان از `F.scaled_dot_product_attention(q, k, v, is_causal=True)` استفاده کنید (که روی GPU از Flash Attention بهره می‌برد) و سرعت آموزش را مقایسه کنید.

</div>

In [ ]:
STEPS = 300
for fused in (False, True):
    torch.manual_seed(0)
    model = MiniGPT(len(corpus.chars), fused=fused)
    seconds = train_gpt(model, corpus, steps=STEPS, log=False)
    print(f'fused={fused!s:5}: {STEPS} steps in {seconds:.1f}s ({1000 * seconds / STEPS:.0f} ms/step) | val loss {val_loss(model, corpus):.3f}')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`F.scaled_dot_product_attention` بسته به سخت‌افزار یکی از چند پیاده‌سازی بهینه را انتخاب می‌کند. روی GPUهای جدید از **Flash Attention** (Dao et al., 2022) استفاده می‌کند. این روش ماتریس $T \times T$ امتیازها را هرگز به‌طور کامل در حافظهٔ اصلی GPU نمی‌سازد و محاسبه را در بلوک‌های کوچک داخل حافظهٔ سریع روی تراشه انجام می‌دهد. نتیجه: حافظه از $O(T^2)$ به $O(T)$ می‌رسد و سرعت چند برابر می‌شود، بدون هیچ تغییری در خروجی ریاضی. روی T4 و با دنبالهٔ کوتاه ۱۲۸، معمولاً تسریع کل آموزش ۱٫۵ تا ۲ برابر است، چون بخشی از زمان صرف لایه‌های دیگر می‌شود. با دنباله‌های بلندتر تفاوت بسیار بیشتر است. Loss نهایی دو مدل عملاً یکی است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

Embedding موقعیت قابل آموزش را با Positional Encoding سینوسی جایگزین کنید (با `register_buffer`). آیا Loss نهایی تغییر می‌کند؟

</div>

In [ ]:
for positional in ('learned', 'sinusoidal'):
    torch.manual_seed(0)
    model = MiniGPT(len(corpus.chars), positional=positional)
    print(f'--- {positional}: {n_params(model):,} trainable parameters, '
          f'{sum(b.numel() for b in model.buffers()):,} values in buffers')
    train_gpt(model, corpus)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`register_buffer` تنسوری را جزء حالت مدل ثبت می‌کند: با `.to(device)` جابه‌جا و در `state_dict` ذخیره می‌شود، اما **پارامتر نیست** و Optimizer آن را تغییر نمی‌دهد. نسخهٔ سینوسی حدود ۲۵ هزار پارامتر کمتر دارد ($128 \times 192$). Loss نهایی دو روش معمولاً بسیار نزدیک است؛ همان نتیجه‌ای که مقالهٔ اصلی Transformer هم گزارش کرد. مزیت نظری روش سینوسی، تعمیم به طول‌های بلندتر از دنباله‌های آموزشی است. در عمل این تعمیم محدود است و مدل‌های جدید از RoPE یا ALiBi استفاده می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

ابعاد مدل را تغییر دهید: یک بار `n_layers=2` و یک بار `d_model=96`. Loss اعتبارسنجی را بر حسب تعداد پارامتر رسم کنید.

</div>

In [ ]:
configs = {'base (6 layers, d=192)': dict(), '2 layers': dict(n_layers=2), 'd_model = 96': dict(d_model=96)}
points = []
for name, kw in configs.items():
    torch.manual_seed(0)
    model = MiniGPT(len(corpus.chars), **kw)
    train_gpt(model, corpus, log=False)
    points.append((n_params(model), val_loss(model, corpus), name))
    print(f'{name:22}: {n_params(model):>9,} parameters | val loss {points[-1][1]:.3f}')

for n, loss, name in points:
    plt.scatter(n, loss, s=60); plt.annotate(name, (n, loss), textcoords='offset points', xytext=(6, 6))
plt.xscale('log'); plt.xlabel('parameters (log scale)'); plt.ylabel('validation loss'); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با بودجهٔ آموزش یکسان، مدل بزرگ‌تر Loss کمتری می‌گیرد. کاهش `d_model` به نصف، پارامترها را تقریباً یک‌چهارم می‌کند، چون بیشتر پارامترها در ماتریس‌های $d \times d$ هستند. کم کردن لایه‌ها پارامترها را تقریباً یک‌سوم می‌کند. این نقاط یک نمونهٔ کوچک از **قوانین مقیاس (Scaling Laws)** هستند (Kaplan et al., 2020): Loss با افزایش پارامتر، داده و محاسبات به‌صورت توانی و قابل پیش‌بینی کم می‌شود. این قانون فقط وقتی برقرار است که داده و زمان آموزش هم متناسب زیاد شوند؛ مدل بزرگ با آموزش ناکافی ممکن است از مدل کوچک بدتر باشد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

**Top-p Sampling** را به متد `generate` اضافه کنید.

</div>

In [ ]:
torch.manual_seed(0)
gpt = MiniGPT(len(corpus.chars))
train_gpt(gpt, corpus, log=False)
gpt.eval()

# what does the nucleus look like for a single step?
with torch.no_grad():
    logits, _ = gpt(torch.tensor([corpus.encode('ROMEO:\nWhat light through yonder ')], device=device))
for p in (0.5, 0.9, 0.99):
    kept = torch.isfinite(top_p_filter(logits[0, -1:].clone(), p)).sum().item()
    print(f'top_p = {p}: {kept} of {len(corpus.chars)} characters kept')

prompt = torch.tensor([corpus.encode('ROMEO:\n')], device=device)
for top_p in (0.5, 0.95):
    torch.manual_seed(0)
    print(f'===== top_p = {top_p} =====')
    print(corpus.decode(gpt.generate(prompt, 250, top_p=top_p)[0].tolist()), '\n')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`top_p_filter` احتمال‌ها را نزولی مرتب می‌کند و جمع تجمعی می‌گیرد. هر Token که **قبل از** آن، جمع به $p$ رسیده باشد حذف می‌شود؛ پس Tokenی که مرز $p$ را رد می‌کند هنوز نگه داشته می‌شود و دست‌کم یک Token همیشه باقی می‌ماند. سپس Logitها با `scatter` به ترتیب اصلی برگردانده می‌شوند.

مزیت Top-p نسبت به Top-k **تطبیقی بودن** است. وقتی مدل مطمئن است (مثلاً وسط یک کلمه)، هسته فقط یکی دو کاراکتر دارد. وقتی مدل نامطمئن است (شروع کلمهٔ جدید)، گزینه‌های بیشتری مجاز می‌شوند. Top-k در هر دو حالت دقیقاً $k$ گزینه نگه می‌دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

مدل را روی یک متن فارسی با مالکیت عمومی (مثلاً غزلیات حافظ یا شاهنامه از گنجور) آموزش دهید. به جای شکسپیر، شعر فارسی تولید کنید!

</div>

In [ ]:
HAFEZ_URL = 'https://raw.githubusercontent.com/PerseusDL/canonical-farsiLit/master/data/hafez/divan/hafez.divan.perseus-far1.xml'
root = ET.fromstring(urllib.request.urlopen(HAFEZ_URL).read())
NS = {'tei': 'http://www.tei-c.org/ns/1.0'}
ghazals = []
for poem in root.iterfind('.//tei:div[@subtype="poem"]', NS):
    lines = [seg.text.strip() for seg in poem.iterfind('.//tei:seg', NS) if seg.text]
    ghazals.append('\n'.join(lines))                    # one hemistich per line
hafez_text = '\n\n'.join(ghazals)                       # a blank line between ghazals
print(f'{len(ghazals)} poems | {len(hafez_text):,} characters')
print(hafez_text[:200])

hafez = CharCorpus(hafez_text)
print('vocabulary:', len(hafez.chars), 'characters')
torch.manual_seed(0)
poet = MiniGPT(len(hafez.chars))
train_gpt(poet, hafez)

poet.eval()
for start in ['دل', 'ساقی']:
    torch.manual_seed(1)
    out = poet.generate(torch.tensor([hafez.encode(start)], device=device), 300, temperature=0.8, top_p=0.95)
    print(f'===== prompt: {start} =====\n{hafez.decode(out[0].tolist())}\n')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

متن دیوان حافظ را از مخزن متن‌باز **Perseus Digital Library** (با مجوز آزاد) در قالب TEI XML می‌خوانیم. هر غزل یک `div` با `subtype="poem"` است و هر مصراع یک عنصر `seg`. هر مصراع را در یک خط و بین غزل‌ها یک خط خالی گذاشتیم تا مدل ساختار شعر را هم یاد بگیرد.

متن حافظ (حدود ۳۰۰ هزار کاراکتر) از شکسپیر کوچک‌تر است؛ پس مدل زودتر بیش‌برازش می‌کند و Loss اعتبارسنجی را باید زیر نظر داشت. خروجی مدل معمولاً کلمات و ترکیب‌های رایج دیوان (می، ساقی، دل، یار)، طول تقریبی مصراع و حتی گاهی **ردیف** و **قافیه** را تقلید می‌کند، اما معنا و وزن عروضی دقیق ندارد. برای بهتر کردن نتیجه، کل اشعار گنجور را اضافه کنید یا از یک مدل زبانی فارسی ازپیش‌آموزش‌دیده شروع کنید (جلسهٔ ۱۴).

</div>